# Model Input Analsysis 

This notebook examines the train, test and validation examples, analyzes persona and sequence lengths.

## Prerequisite

Generate dataset split with compact prompts
```bash
uv run python scripts/preprocess_data.py \
  --data-config configs/data/twin2k500.yaml
```


In [1]:
from datasets import load_from_disk
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import numpy as np
import pandas as pd

In [2]:
participant_splits = load_from_disk(
    "../data/processed/twin2k500_compact"
)

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    use_fast=True,
)

In [3]:
from pathlib import Path

from datasets import DatasetDict, load_from_disk

PROMPT_DATA_DIR = Path(
    "../data/processed/prompts/qwen25_05b_compact_max"
)

if not PROMPT_DATA_DIR.is_dir():
    raise FileNotFoundError(
        "Prepared prompt data was not found at "
        f"{PROMPT_DATA_DIR.resolve()}. "
        "Run scripts/prepare_prompt_data.py first."
    )

prompt_splits = load_from_disk(
    PROMPT_DATA_DIR
)

if not isinstance(prompt_splits, DatasetDict):
    raise TypeError(
        f"Expected a DatasetDict, received "
        f"{type(prompt_splits).__name__}."
    )

prompt_splits

DatasetDict({
    train: Dataset({
        features: ['pid', 'block_name', 'question_id', 'question_type', 'prompt_text', 'target_text'],
        num_rows: 90720
    })
    validation: Dataset({
        features: ['pid', 'block_name', 'question_id', 'question_type', 'prompt_text', 'target_text'],
        num_rows: 12915
    })
    test: Dataset({
        features: ['pid', 'block_name', 'question_id', 'question_type', 'prompt_text', 'target_text'],
        num_rows: 26019
    })
})

In [4]:
split_names = [
    "train",
    "validation",
    "test",
]

total_participants = sum(
    len(set(participant_splits[name]["pid"]))
    for name in split_names
)

records = []

for split_name in split_names:
    participant_count = len(
        set(participant_splits[split_name]["pid"])
    )
    example_count = len(
        prompt_splits[split_name]
    )

    records.append(
        {
            "split": split_name.title(),
            "participants": participant_count,
            "examples": example_count,
            "percentage": (
                100
                * participant_count
                / total_participants
            ),
        }
    )

split_summary = pd.DataFrame(records)

split_summary

,split,participants,examples,percentage
0,Train,1440,90720,69.970845
1,Validation,205,12915,9.961127
2,Test,413,26019,20.068027


## Persona compression

In [5]:
def persona_token_lengths(
    texts,
    *,
    batch_size: int = 16,
    description: str = "Tokenizing personas",
) -> np.ndarray:
    lengths = []

    with tqdm(
        total=len(texts),
        desc=description,
        unit="persona",
    ) as progress:
        for start in range(
            0,
            len(texts),
            batch_size,
        ):
            batch = texts[
                start : start + batch_size
            ]

            encoded = tokenizer(
                batch,
                add_special_tokens=False,
                return_length=True,
                truncation=False,
            )

            lengths.extend(encoded["length"])
            progress.update(len(batch))

    return np.asarray(
        lengths,
        dtype=np.int64,
    )

In [6]:
records = []

for split_name in [
    "train",
    "validation",
    "test",
]:
    split = participant_splits[split_name]

    raw_lengths = persona_token_lengths(
        split["wave1_3_persona_text"],
        description=f"{split_name}: raw personas",
    )

    compact_lengths = persona_token_lengths(
        split["wave1_3_compact_persona_text"],
        description=f"{split_name}: compact personas",
    )

    records.append(
        {
            "split": split_name,
            "participants": len(split),
            "raw_tokens_mean": raw_lengths.mean(),
            "raw_tokens_median": np.median(
                raw_lengths
            ),
            "raw_tokens_max": raw_lengths.max(),
            "compact_tokens_mean": (
                compact_lengths.mean()
            ),
            "compact_tokens_median": np.median(
                compact_lengths
            ),
            "compact_tokens_max": (
                compact_lengths.max()
            ),
            "mean_reduction_pct": (
                100
                * (
                    1
                    - compact_lengths
                    / raw_lengths
                ).mean()
            ),
        }
    )

persona_summary = (
    pd.DataFrame(records)
    .set_index("split")
    .round(1)
)

persona_summary

train: raw personas:   0%|          | 0/1440 [00:00<?, ?persona/s]

train: compact personas:   0%|          | 0/1440 [00:00<?, ?persona/s]

validation: raw personas:   0%|          | 0/205 [00:00<?, ?persona/s]

validation: compact personas:   0%|          | 0/205 [00:00<?, ?persona/s]

test: raw personas:   0%|          | 0/413 [00:00<?, ?persona/s]

test: compact personas:   0%|          | 0/413 [00:00<?, ?persona/s]

,participants,raw_tokens_mean,raw_tokens_median,raw_tokens_max,compact_tokens_mean,compact_tokens_median,compact_tokens_max,mean_reduction_pct
split,,,,,,,,
train,1440,27526.6,27512.5,28760,16587.0,16571.5,17816,39.7
validation,205,27520.3,27515.0,27962,16582.2,16580.0,17096,39.7
test,413,27526.1,27518.0,27932,16590.9,16572.0,16999,39.7
